In [ ]:
import Implementation_OFLP as IO
import run_OFLP as rO
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import numpy as np
import math
a = rO.read_instance('ofl_12_100i.gms')

def get_feas_points(circles, x_lb, x_ub, y_lb, y_ub) -> float:
    # We create 1000 possible facility locations, we then check for each which are feasible to get points to try to fit the number of facilities.
    poss_fac = []
    x_coords = np.linspace(x_lb,x_ub,50)
    y_coords = np.linspace(y_lb,y_ub,50)
    for x in x_coords:
        for y in y_coords:
            p = [x,y]
            feas = True
            for c in circles:
                if IO.interior_circle(p,c):
                    feas = False
                    break
            if feas:
                poss_fac.append(p)
    return poss_fac

def find_n_clique(points, fac_fac_dist, n) -> list:
    '''
    This definition finds an n-clique in a graph G. If it finds one it terminates immediatly and returns it.
    :param G: The graph that is considered.
    :param n: The size of the clique that is being searched for.
    '''
    if n <= 0:
        return []
    if n == 1:
        return [0] if len(points) > 0 else []
    if len(points) < n:
        return []
    
    G = nx.Graph()
    pts = np.array(points)
    d = ((pts[:, None, :] - pts[None, :, :]) ** 2).sum(axis=2)
    ii, jj = np.where(np.triu(d >= fac_fac_dist, k=1))
    G.add_edges_from(zip(ii.tolist(), jj.tolist()))
    G = nx.k_core(G, n - 1)

    for v in G:
        if G.degree(v) < n - 1:
            continue
        else:
            clique = [v]
            neighbours = []
            for w in G.neighbors(v):
                if w > v:
                    neighbours.append(w)
            level_neighbours = [neighbours]
            clique_explored = False
            while not clique_explored:
                if len(clique) == n:
                    return clique
                if len(clique) + len(level_neighbours[-1]) < n:
                    clique.pop()
                    level_neighbours.pop()
                    if len(clique) == 0:
                        clique_explored = True
                else:
                    w = level_neighbours[-1].pop()
                    if G.degree(w) < n - 1:
                        continue
                    poss_vert = []
                    for wi in level_neighbours[-1]:
                        if G.has_edge(w, wi):
                            poss_vert.append(wi)
                    clique.append(w)
                    level_neighbours.append(poss_vert)
    return []

def create_init_sol(facilities, communities, x_lb, x_ub, y_lb, y_ub, fac_fac_dist, r_guess : float=0) -> float:
    # The idea of the algorithm is to decrease the radius until we find a feasible radius, this way we create a good intial solution.
    feas_sol_found = False
    max_iter = 1e3 * len(communities)
    t = 0
    if r_guess == 0:
        r = 2.5 / math.sqrt(len(communities))
    else:
        r = r_guess
    with tqdm(total=max_iter, desc="Shrinking r") as pbar:
        while (not feas_sol_found) and (t < max_iter):
            circles = []
            for i in communities:
                circles.append([i[0], i[1], r])
            points = get_feas_points(circles, x_lb, x_ub, y_lb, y_ub)
            # We now try to fit the facilities in the points while maintaining the fac_fac_distance
            clique = find_n_clique(points, fac_fac_dist, facilities)
            if clique == []:
                r *= 0.95
                pbar.update(1)
            else:
                feas_sol_found = True
                fig, ax = plt.subplots()
                box = patches.Rectangle((x_lb, y_lb), x_ub-x_lb, y_ub - y_lb, linewidth=3, edgecolor='black', facecolor='none')
                ax.add_patch(box)
                for c in circles:
                    circle = patches.Circle((c[0], c[1]), c[2], linewidth=1, edgecolor='black', fill = True)
                    ax.add_patch(circle)
                for p in clique:
                    ax.scatter(points[p][0], points[p][1], s=20, c='red', marker='o')
                # Set limits and equal aspect ratio
                ax.set_xlim(x_lb - abs(x_lb-x_ub)/2, x_ub + abs(x_lb-x_ub)/2)
                ax.set_ylim(y_lb - abs(y_lb-y_ub)/2, y_ub + abs(y_lb-y_ub)/2)
                ax.set_aspect('equal')
            
                # Show the plot
                plt.grid(True)
                plt.show()
                return r, clique
    return r, clique

print(a['ff_sq_bounds'])
create_init_sol(a['facilities'], a['communities'], a['x_bounds'][0], a['x_bounds'][1], a['y_bounds'][0], a['y_bounds'][1], a['ff_sq_bounds'][0])

(0.041666666666666664, 2.0)


Shrinking r:   0%|          | 18/100000.0 [02:15<260:28:40,  9.38s/it]